In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("artists.csv")

# Keep only rows where ALL columns have data
complete_all = df.dropna(subset=[
    "Artist ID",
    "Name",
    "Nationality",
    "Gender",
    "Birth Year",
    "Death Year"
])

# Save new spreadsheet
complete_all.to_csv("artists_all_fields_complete.csv", index=False)

print(f"Number of complete records: {len(complete_all)}")
print("File created: artists_all_fields_complete.csv")

Number of complete records: 4429
File created: artists_all_fields_complete.csv


In [2]:
import pandas as pd

df = pd.read_csv("artists.csv")

likely_alive = df[
    (df["Death Year"].isna()) &
    (df["Birth Year"] >= 1960)
]

likely_alive.to_csv("artists_likely_alive.csv", index=False)

print(f"Number of likely living artists: {len(likely_alive)}")
print("File created: artists_likely_alive.csv")

Number of likely living artists: 2349
File created: artists_likely_alive.csv


In [3]:
import pandas as pd

df = pd.read_csv("artists.csv")

missing_data = df[
    df["Artist ID"].isna() |
    df["Name"].isna() |
    df["Nationality"].isna() |
    df["Gender"].isna() |
    df["Birth Year"].isna() |
    df["Death Year"].isna()
]

missing_data.to_csv("artists_missing_any_field.csv", index=False)

print(f"Number of incomplete records: {len(missing_data)}")
print("File created: artists_missing_any_field.csv")

Number of incomplete records: 10662
File created: artists_missing_any_field.csv


In [4]:
import pandas as pd

df = pd.read_csv("artists.csv")

df["wikidata_id"] = ""
df["enhanced_birth_year"] = ""
df["match_status"] = ""

df.to_csv("artists_prepared_for_wikidata.csv", index=False)

print("File created: artists_prepared_for_wikidata.csv")

File created: artists_prepared_for_wikidata.csv


In [5]:
import pandas as pd
import requests
import time

In [6]:
df = pd.read_csv("artists_prepared_for_wikidata.csv")

print(df.head())

   Artist ID             Name Nationality Gender  Birth Year  Death Year  \
0          1   Robert Arneson    American   Male      1930.0      1992.0   
1          2   Doroteo Arnaiz     Spanish   Male      1936.0         NaN   
2          3      Bill Arnold    American   Male      1941.0         NaN   
3          4  Charles Arnoldi    American   Male      1946.0         NaN   
4          5      Per Arnoldi      Danish   Male      1941.0         NaN   

   wikidata_id  enhanced_birth_year  match_status  
0          NaN                  NaN           NaN  
1          NaN                  NaN           NaN  
2          NaN                  NaN           NaN  
3          NaN                  NaN           NaN  
4          NaN                  NaN           NaN  


In [7]:
def search_wikidata(name):

    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json",
        "limit": 5
    }

    response = requests.get(url, params=params)

    return response.json()["search"]

In [9]:
def get_entity(entity_id):

    url = "https://www.wikidata.org/wiki/Special:EntityData/" + entity_id + ".json"

    response = requests.get(url)

    return response.json()

In [10]:
def compare_artist(candidate, original_birth):

    description = candidate.get("description","").lower()

    if "artist" not in description and \
       "painter" not in description and \
       "sculptor" not in description and \
       "photographer" not in description and \
       "designer" not in description:

        return False

    if pd.notna(original_birth):

        try:

            candidate_birth = candidate.get("display",{}).get("birthYear")

            if candidate_birth:

                if abs(int(candidate_birth)-int(original_birth)) > 3:

                    return False

        except:

            pass

    return True

In [11]:
for index,row in df.iterrows():

    name = row["Name"]

    birth = row["Birth Year"]

    if pd.isna(name):

        df.at[index,"match_status"]="not_found"

        continue

    results = search_wikidata(name)

    if len(results)==0:

        df.at[index,"match_status"]="not_found"

        continue

    if len(results)==1:

        candidate = results[0]

        df.at[index,"wikidata_id"]=candidate["id"]

        df.at[index,"match_status"]="matched"

    else:

        matched=False

        for candidate in results:

            if compare_artist(candidate,birth):

                df.at[index,"wikidata_id"]=candidate["id"]

                df.at[index,"match_status"]="matched"

                matched=True

                break

        if matched==False:

            df.at[index,"match_status"]="uncertain"

    time.sleep(0.2)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [12]:
import json
import requests
import time
import pandas as pd

for index, row in df.iterrows():
    name = row["Name"]
    birth = row["Birth Year"]
    
    if pd.isna(name):
        df.at[index, "match_status"] = "not_found"
        continue
    
    try:
        # Add error handling for the search_wikidata function
        results = search_wikidata(name)
        
        # Check if results is None or empty due to API errors
        if results is None:
            df.at[index, "match_status"] = "api_error"
            continue
            
    except (json.JSONDecodeError, requests.exceptions.RequestException) as e:
        # Handle JSON decode errors and network issues
        print(f"Error processing {name}: {e}")
        df.at[index, "match_status"] = "api_error"
        continue
    
    if len(results) == 0:
        df.at[index, "match_status"] = "not_found"
        continue
    
    if len(results) == 1:
        candidate = results[0]
        df.at[index, "wikidata_id"] = candidate["id"]
        df.at[index, "match_status"] = "matched"
    else:
        matched = False
        for candidate in results:
            if compare_artist(candidate, birth):
                df.at[index, "wikidata_id"] = candidate["id"]
                df.at[index, "match_status"] = "matched"
                matched = True
                break
        
        if matched == False:
            df.at[index, "match_status"] = "uncertain"
    
    time.sleep(0.2)  # Keep the delay to avoid overwhelming the API


Error processing Robert Arneson: Expecting value: line 1 column 1 (char 0)


/var/folders/lc/mggnjvs146b_n1rwwpvgdrlc0000gn/T/ipykernel_69382/811744969.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'api_error' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, "match_status"] = "api_error"


Error processing Doroteo Arnaiz: Expecting value: line 1 column 1 (char 0)
Error processing Bill Arnold: Expecting value: line 1 column 1 (char 0)
Error processing Charles Arnoldi: Expecting value: line 1 column 1 (char 0)
Error processing Per Arnoldi: Expecting value: line 1 column 1 (char 0)
Error processing Danilo Aroldi: Expecting value: line 1 column 1 (char 0)
Error processing Bill Aron: Expecting value: line 1 column 1 (char 0)
Error processing David Aronson: Expecting value: line 1 column 1 (char 0)
Error processing Irene Aronson: Expecting value: line 1 column 1 (char 0)
Error processing Jean (Hans) Arp: Expecting value: line 1 column 1 (char 0)
Error processing Jüri Arrak: Expecting value: line 1 column 1 (char 0)
Error processing J. Arrelano Fischer: Expecting value: line 1 column 1 (char 0)
Error processing Folke Arstrom: Expecting value: line 1 column 1 (char 0)
Error processing Cristobal Arteche: Expecting value: line 1 column 1 (char 0)
Error processing Artko: Expecting 

In [13]:
df.to_csv("artists_enhanced_wikidata.csv",index=False)

print("Finished!")

print(df["match_status"].value_counts())

Finished!
match_status
api_error    15091
Name: count, dtype: int64
